Definimos que vamos a obtener

In [16]:
from pprint import pprint

url = "https://web.archive.org/web/20201112014336if_/http://www.almhuette-raith.at/apache-log/access.log"
destination = "tmpdir/logfile.txt"

Generamos los folders intermedios que necesitamos (esta explicitamente en el gitignore)

In [17]:
from pathlib import Path

full_path = Path(destination)
directory = full_path.parent

# Creates the directory and any missing parent folders
Path(directory).mkdir(parents=True, exist_ok=True)

obtenemos el archivo con el que vamos a trabajar.

Partimos de un codigo robusto, pero sin controles de edad.

In [18]:
import requests


if not full_path.is_file():
# Streams the file to handle large downloads efficiently without overloading memory
    with requests.get(url, stream=True) as response:
        response.raise_for_status() # Throws an error for bad responses (404, 500, etc.)
        with open(destination, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

Exploramos inicialmente el archivo

In [19]:
sample_log = None
with open(destination, "r") as file:
    for _ in range(10):
        line = file.readline()
        if not line:
            break  # Stops early if the file has fewer than 10 lines
        print(line, end="")
        sample_log = line


109.169.248.247 - - [12/Dec/2015:18:25:11 +0100] "GET /administrator/ HTTP/1.1" 200 4263 "-" "Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0" "-"
109.169.248.247 - - [12/Dec/2015:18:25:11 +0100] "POST /administrator/index.php HTTP/1.1" 200 4494 "http://almhuette-raith.at/administrator/" "Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0" "-"
46.72.177.4 - - [12/Dec/2015:18:31:08 +0100] "GET /administrator/ HTTP/1.1" 200 4263 "-" "Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0" "-"
46.72.177.4 - - [12/Dec/2015:18:31:08 +0100] "POST /administrator/index.php HTTP/1.1" 200 4494 "http://almhuette-raith.at/administrator/" "Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0" "-"
83.167.113.100 - - [12/Dec/2015:18:31:25 +0100] "GET /administrator/ HTTP/1.1" 200 4263 "-" "Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0" "-"
83.167.113.100 - - [12/Dec/2015:18:31:25 +0100] "POST /administrator/index.php HTTP/1

Establescamos un datamodel

In [20]:
from pydantic import BaseModel, Field, IPvAnyAddress, HttpUrl
from datetime import datetime
from typing import Optional

# 1. Define the Pydantic schema
class ApacheLog(BaseModel):
    ip_address: IPvAnyAddress
    client_identity: Optional[str] = None
    userid: Optional[str] = None
    timestamp: datetime
    method: str
    request_uri: str
    http_version: str
    status_code: int = Field(ge=100, le=599)
    userAgent: Optional[str] = None
    referer: Optional[str] = None
    response_size: Optional[int] = None

Definamos nuestra expresion regular

In [21]:

regex = '^([(\d\.)]+) [^ ]* [^ ]* \[([^ ]* [^ ]*)\] "([^"]*)" (\d+) [^ ]* "([^"]*)" "([^"]*)"'

HOST = r'^(?P<host>.*?)'
SPACE = r'\s'
IDENTITY = r'\S+'
USER = r'\S+'
TIME = r'(?P<time>\[.*?\])'
REQUEST = r'\"(?P<request>.*?)\"'
STATUS = r'(?P<status>\d{3})'
SIZE = r'(?P<size>\S+)'

full_regex = HOST+SPACE+IDENTITY+SPACE+USER+SPACE+TIME+SPACE+REQUEST+SPACE+STATUS+SPACE+SIZE+SPACE



Mapeamos la linea de ejemplo que traemos

In [22]:
import re
def parse_apache_line_regex_initial(line: str) -> Optional[ApacheLog]:
    initial_parse = re.match(regex, line).groups()
    request_elements = initial_parse[2].split()
    return ApacheLog(
            ip_address=initial_parse[0],
            client_identity=None,
            userid=None,
            timestamp=datetime.strptime(initial_parse[1], "%d/%b/%Y:%H:%M:%S %z"),
            method=request_elements[0],
            request_uri=request_elements[1],
            http_version=request_elements[2],
            status_code=int(initial_parse[3]),
            response_size=None,
            userAgent=initial_parse[5]
        )
        
def parse_apache_line_regex_next(line: str) -> Optional[ApacheLog]:
    match = re.search(full_regex,line)
    request_elements = match.group('request').split()
    return ApacheLog(
            ip_address=match.group('host'),
            client_identity=None,
            userid=None,
            timestamp=datetime.strptime(match.group('time'), "[%d/%b/%Y:%H:%M:%S %z]"),
            method=request_elements[0],
            request_uri=request_elements[1],
            http_version=request_elements[2],
            status_code=int(match.group('status')),
            response_size=match.group('size')
        )


print(re.match(regex, sample_log).groups())
print(parse_apache_line_regex_initial(sample_log))
print(re.match(full_regex, sample_log).groups())
print(parse_apache_line_regex_next(sample_log))

('109.184.11.34', '12/Dec/2015:18:32:56 +0100', 'GET /administrator/ HTTP/1.1', '200', '-', 'Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0')
ip_address=IPv4Address('109.184.11.34') client_identity=None userid=None timestamp=datetime.datetime(2015, 12, 12, 18, 32, 56, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))) method='GET' request_uri='/administrator/' http_version='HTTP/1.1' status_code=200 userAgent='Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0' referer=None response_size=None
('109.184.11.34', '[12/Dec/2015:18:32:56 +0100]', 'GET /administrator/ HTTP/1.1', '200', '4263')
ip_address=IPv4Address('109.184.11.34') client_identity=None userid=None timestamp=datetime.datetime(2015, 12, 12, 18, 32, 56, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))) method='GET' request_uri='/administrator/' http_version='HTTP/1.1' status_code=200 userAgent=None referer=None response_size=4263


via cuts

In [23]:
def parse_apache_line(line: str) -> Optional[ApacheLog]:
    try:
        # Split into main structural parts based on quotes and brackets
        parts = line.split(' "')
        if len(parts) < 2:
            return None
        
        front = parts[0].split(' ')
        req_and_back = parts[1].split('" ')
        
        ip = front[0]
        ident = front[1] if front[1] != '-' else None
        user = front[2] if front[2] != '-' else None
        
        # Strip brackets from timestamp
        time_str = front[3][1:] + " " + front[4][:-1] 
        # Parse Apache's specific time format
        dt = datetime.strptime(time_str, "%d/%b/%Y:%H:%M:%S %z")
        
        req_parts = req_and_back[0].split(' ')
        method = req_parts[0]
        uri = req_parts[1]
        version = req_parts[2]
        
        status_and_size = req_and_back[1].split(' ')
        status = int(status_and_size[0])
        size = int(status_and_size[1]) if status_and_size[1].isdigit() else None

        return ApacheLog(
            ip_address=ip,
            client_identity=ident,
            userid=user,
            timestamp=dt,
            method=method,
            request_uri=uri,
            http_version=version,
            status_code=status,
            response_size=size
        )
    except Exception as e:
        print(f"Failed to parse line: {e}")
        return None

print(parse_apache_line(sample_log))

ip_address=IPv4Address('109.184.11.34') client_identity=None userid=None timestamp=datetime.datetime(2015, 12, 12, 18, 32, 56, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))) method='GET' request_uri='/administrator/' http_version='HTTP/1.1' status_code=200 userAgent=None referer=None response_size=4263


Usando un metodo que si funciona... (pero solo en este caso...)

In [24]:

import apachelogs


def parse_apachelogs(line: str) -> Optional[ApacheLog]:
    try:
        log_format = '%h %l %u %t "%r" %s %b "%{Referer}i" "%{User-Agent}i" "%u"'
        parser = apachelogs.LogParser(log_format,errors=None)
        log_line_data = parser.parse(line)
        request_elements = log_line_data.request_line.split()
        return ApacheLog(
            ip_address=log_line_data.remote_host,
            client_identity=log_line_data.remote_logname,
            userid=log_line_data.remote_user,
            timestamp=log_line_data.request_time,
            method=request_elements[0],
            request_uri=request_elements[1],
            http_version=request_elements[2],
            status_code=log_line_data.status,
            userAgent=log_line_data.headers_in["User-Agent"],
            referer=log_line_data.headers_in["Referer"],
            response_size=log_line_data.bytes_sent
            
        )
    except Exception as e:
        print(f"Failed to parse line: {e}")
        return None

pprint(parse_apachelogs(sample_log))

ApacheLog(ip_address=IPv4Address('109.184.11.34'), client_identity=None, userid=None, timestamp=datetime.datetime(2015, 12, 12, 18, 32, 56, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), method='GET', request_uri='/administrator/', http_version='HTTP/1.1', status_code=200, userAgent='Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0', referer=None, response_size=4263)


Buscamos extender datos

In [25]:
import socket
import functools

@functools.lru_cache(maxsize=3200)
def reverse_dns_lookup(ip_address):
    try:
        # Returns a tuple: (hostname, aliaslist, ipaddrlist)
        hostname, _, _ = socket.gethostbyaddr(ip_address)
        return hostname
    except socket.herror:
        return "Unknown host (No PTR record found)"
    except Exception as e:
        return f"An error occurred: {e}"

# Example usage
ip = "8.8.8.8"
print(f"IP: {ip} -> Hostname: {reverse_dns_lookup(ip)}")

IP: 8.8.8.8 -> Hostname: dns.google


Como extendemos el data model?

In [26]:
class Apachelog_reversedns(ApacheLog):
    reverse_dns_lookup: Optional[str] = None

log_register = parse_apachelogs(sample_log)

log_register_reversed_dns = Apachelog_reversedns(** log_register.model_dump(),
     reverse_dns_lookup=reverse_dns_lookup(str(log_register.ip_address)))


print(log_register_reversed_dns)

ip_address=IPv4Address('109.184.11.34') client_identity=None userid=None timestamp=datetime.datetime(2015, 12, 12, 18, 32, 56, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))) method='GET' request_uri='/administrator/' http_version='HTTP/1.1' status_code=200 userAgent='Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0' referer=None response_size=4263 reverse_dns_lookup='109-184-11-34.dynamic.mts-nn.ru'


Perpo, podemos hacerlo mas automatico?

In [27]:
from pydantic import computed_field
class Apachelog_autoreversedns(ApacheLog):
    reverse_dns_lookup: Optional[str] = None

    @computed_field
    def reverse_dns(self) -> str:
        return reverse_dns_lookup(str(self.ip_address))


log_register_auto_reversed_dns = Apachelog_autoreversedns(** log_register.model_dump())


print(log_register_auto_reversed_dns)

ip_address=IPv4Address('109.184.11.34') client_identity=None userid=None timestamp=datetime.datetime(2015, 12, 12, 18, 32, 56, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))) method='GET' request_uri='/administrator/' http_version='HTTP/1.1' status_code=200 userAgent='Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20100101 Firefox/34.0' referer=None response_size=4263 reverse_dns_lookup=None reverse_dns='109-184-11-34.dynamic.mts-nn.ru'


Creamos un wraper para obtener archivos

In [28]:
from pathlib import Path
import requests

def get_file(url: str, destination: str) -> None:
    full_path = Path(destination)
    directory = full_path.parent
    # Creates the directory and any missing parent folders
    # Path(directory).mkdir(parents=True, exist_ok=True)
    if not full_path.is_file():
    # Streams the file to handle large downloads efficiently without overloading memory
        with requests.get(url, stream=True) as response:
            response.raise_for_status() # Throws an error for bad responses (404, 500, etc.)
            with open(destination, "wb") as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
    else:
        print(f"The file {destination} is preview downloaded")

get_file(url,destination)

http_status_catalog_url = "https://gist.githubusercontent.com/josantonius/0a889ab6f18db2fcefda15a039613293/raw/346e06dceb6ad1309b88c68046d634bfe125bdf9/http-status-codes.json"
http_status_catalog_file ="tmpdir/status.json"
get_file(http_status_catalog_url, http_status_catalog_file)


The file tmpdir/logfile.txt is preview downloaded
The file tmpdir/status.json is preview downloaded


Los estados http son correctos?

In [34]:
import json

class HttpStatusValidator(BaseModel):
    http_status: int = Field(ge=100, le=599)
    http_description: Optional[str]
    is_valid: bool


# Open the file and decode the JSON data
with open(http_status_catalog_file, 'r', encoding='utf-8') as file:
    http_status_catalog = json.load(file)


def validate_http_status(status: int) -> HttpStatusValidator:
    if str(status) in http_status_catalog["en"]:
        return HttpStatusValidator(
            http_status = status,
            http_description = http_status_catalog["en"][str(status)]["large"],
            is_valid = True
        )
    else:
        return HttpStatusValidator(
            http_status = status,
            http_description = "unknow status code",
            is_valid = False
        ) 

print(validate_http_status(200))

print(http_status_catalog["en"].get(str(200), {"large": "Incorrect status"})["large"])
print(http_status_catalog["en"].get(str(299), {"large": "Incorrect status"})["large"])

http_status=200 http_description='Standard response for successful HTTP requests. The actual response will depend on the request method used. In a GET request, the response will contain an entity corresponding to the requested resource. In a POST request, the response will contain an entity describing or containing the result of the action.' is_valid=True
Standard response for successful HTTP requests. The actual response will depend on the request method used. In a GET request, the response will contain an entity corresponding to the requested resource. In a POST request, the response will contain an entity describing or containing the result of the action.
Incorrect status


Bueno, y de que pais es el sujeto?

In [ ]:

def verify_ip(ip: IPvAnyAddress):
    payload = f"http://ip-api.com/json/{query}?fields=status,country,countryCode,isp,mobile,proxy,hosting,query"